# 02 – Baseline model

Template: get an honest baseline in ~15 minutes, then iterate.

1. Pick target + features
2. Split **by time** if data is chronological (no future leakage!)
3. Compare against a dumb baseline — judges ask *"better than what?"*
4. Feature importance → story for the presentation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier, XGBRegressor

from app.services.datasets import list_datasets, load_df

[d.id for d in list_datasets()]

## 1. Load + choose target

In [ ]:
df = load_df("_sample/matches.parquet")  # <- change

# Example target (classification): did the home team win?
df["target"] = (df["home_goals"] > df["away_goals"]).astype(int)

TARGET = "target"
TASK = "classification"  # or "regression"
TIME_COL = "date"  # None if not chronological
DROP = ["match_id", "home_goals", "away_goals"]  # ids + anything that leaks the answer!

df.head()

## 2. Split (time-aware) + preprocessing

In [ ]:
data = df.sort_values(TIME_COL) if TIME_COL else df
X = data.drop(columns=[TARGET, *DROP, *([TIME_COL] if TIME_COL else [])])
y = data[TARGET]

if TIME_COL:  # train on past, test on future
    cut = int(len(data) * 0.8)
    X_train, X_test, y_train, y_test = X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y if TASK == "classification" else None
    )

num = X.select_dtypes("number").columns.tolist()
cat = X.select_dtypes(exclude="number").columns.tolist()
pre = ColumnTransformer(
    [
        ("num", SimpleImputer(strategy="median"), num),
        (
            "cat",
            Pipeline(
                [
                    ("imp", SimpleImputer(strategy="most_frequent")),
                    ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
                ]
            ),
            cat,
        ),
    ]
)
print(f"train={len(X_train)} test={len(X_test)} num={num} cat={cat}")

## 3. Baseline vs model

In [ ]:
if TASK == "classification":
    models = {
        "baseline (majority)": DummyClassifier(),
        "xgboost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05),
    }
else:
    models = {
        "baseline (mean)": DummyRegressor(),
        "xgboost": XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05),
    }

results, fitted = [], {}
for name, m in models.items():
    pipe = Pipeline([("pre", pre), ("model", m)]).fit(X_train, y_train)
    fitted[name] = pipe
    pred = pipe.predict(X_test)
    if TASK == "classification":
        proba = pipe.predict_proba(X_test)[:, 1] if y.nunique() == 2 else None
        results.append(
            {
                "model": name,
                "accuracy": accuracy_score(y_test, pred),
                "f1": f1_score(y_test, pred, average="weighted"),
                "roc_auc": roc_auc_score(y_test, proba)
                if proba is not None and y_test.nunique() > 1
                else np.nan,
            }
        )
    else:
        results.append(
            {"model": name, "mae": mean_absolute_error(y_test, pred), "r2": r2_score(y_test, pred)}
        )
pd.DataFrame(results).set_index("model").round(3)

## 4. What drives the prediction? (presentation material)

In [ ]:
best = fitted["xgboost"]
imp = permutation_importance(best, X_test, y_test, n_repeats=10, random_state=42)
importance = pd.Series(imp.importances_mean, index=X_test.columns).sort_values()
importance.plot.barh(figsize=(6, max(2, len(importance) * 0.3)), title="Permutation importance")
plt.tight_layout()

## 5. Save for the API

The backend can load this with `joblib.load` in a service and expose predictions via an endpoint.

In [ ]:
# import joblib
# from app.core.config import settings
# (settings.data_dir.parent / "models").mkdir(exist_ok=True)
# joblib.dump(best, settings.data_dir.parent / "models" / "model.joblib")